# 👖 Autoencoders on Fashion MNIST

In this notebook, we'll walk through the steps required to train your own autoencoder on the fashion MNIST dataset.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
import os
os.environ["KERAS_BACKEND"] = "torch"  # Keras only loads the dataset; this keeps TensorFlow out of the PyTorch process
from keras import datasets

import sys
from pathlib import Path

# Make the repo root importable so `notebooks.utils_pytorch` can be found
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils_pytorch import display

# Check GPU - PyTorch needs to be told which device to use
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # Apple Silicon
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 32
CHANNELS = 1
BATCH_SIZE = 100
BUFFER_SIZE = 1000
VALIDATION_SPLIT = 0.2
EMBEDDING_DIM = 2
EPOCHS = 3

## 1. Prepare the data <a name="prepare"></a>

In [ ]:
# Load the data - Keras downloads it once (cached in ~/.keras/datasets)
# and returns NumPy arrays: images (N, 28, 28) uint8, labels (N,)
(x_train, y_train), (x_test, y_test) = datasets.fashion_mnist.load_data()

In [ ]:
# Preprocess the data


def preprocess(imgs):
    """
    Normalize and reshape the images
    """
    imgs = imgs.astype("float32") / 255.0
    imgs = np.pad(imgs, ((0, 0), (2, 2), (2, 2)), constant_values=0.0)  # 28x28 -> 32x32
    imgs = np.expand_dims(imgs, 1)  # add the channel axis first: (N, 1, H, W)
    return torch.from_numpy(imgs)


x_train = preprocess(x_train)
x_test = preprocess(x_test)

In [ ]:
# Show some items of clothing from the training set
display(x_train)

## 2. Build the autoencoder <a name="build"></a>

In [ ]:
# Encoder
# padding=1 with a 3x3 kernel and stride 2 halves the size, like Keras' padding="same"
encoder = nn.Sequential(
    nn.Conv2d(CHANNELS, 32, kernel_size=3, stride=2, padding=1),  # -> (32, 16, 16)
    nn.ReLU(),
    nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # -> (64, 8, 8)
    nn.ReLU(),
    nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # -> (128, 4, 4)
    nn.ReLU(),
)

# PyTorch layers don't track shapes like Keras, so we push a dummy
# image through the conv layers to find the shape the decoder will need
with torch.no_grad():
    shape_before_flattening = encoder(torch.zeros(1, CHANNELS, IMAGE_SIZE, IMAGE_SIZE)).shape[1:]
print(f"shape_before_flattening: {tuple(shape_before_flattening)}")

encoder.append(nn.Flatten())
encoder.append(nn.Linear(int(np.prod(shape_before_flattening)), EMBEDDING_DIM))
encoder = encoder.to(device)

print(encoder)
print(f"Total params: {sum(p.numel() for p in encoder.parameters()):,}")

In [ ]:
# Decoder
# ConvTranspose2d with stride 2 doubles the size; output_padding=1 is needed
# to get exactly 2x (e.g. 4 -> 8) - the equivalent of Keras' padding="same"
decoder = nn.Sequential(
    nn.Linear(EMBEDDING_DIM, int(np.prod(shape_before_flattening))),
    nn.Unflatten(1, shape_before_flattening),  # -> (128, 4, 4)
    nn.ConvTranspose2d(128, 128, kernel_size=3, stride=2, padding=1, output_padding=1),  # -> (128, 8, 8)
    nn.ReLU(),
    nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),  # -> (64, 16, 16)
    nn.ReLU(),
    nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # -> (32, 32, 32)
    nn.ReLU(),
    nn.Conv2d(32, CHANNELS, kernel_size=3, stride=1, padding=1),  # -> (1, 32, 32)
    nn.Sigmoid(),  # pixel values in [0, 1]
).to(device)

print(decoder)
print(f"Total params: {sum(p.numel() for p in decoder.parameters()):,}")

In [ ]:
# Autoencoder
# The encoder and decoder are shared: training the autoencoder trains both
autoencoder = nn.Sequential(encoder, decoder)

print(autoencoder)
print(f"Total params: {sum(p.numel() for p in autoencoder.parameters()):,}")

## 3. Train the autoencoder <a name="train"></a>

In [ ]:
# Compile the autoencoder
opt = torch.optim.Adam(autoencoder.parameters())  # default lr=0.001, as in Keras
# The decoder already ends with a sigmoid, so we use plain BCELoss
# (BCEWithLogitsLoss would expect raw, pre-sigmoid outputs)
loss_fn = nn.BCELoss()

In [ ]:
# Create a model save checkpoint
# (we save the best weights by hand in the training loop below)
checkpoint_path = "./checkpoint/autoencoder.pt"
best_loss = float("inf")

writer = SummaryWriter(log_dir="./logs")  # view with: tensorboard --logdir ./logs

In [ ]:
train_loader = DataLoader(TensorDataset(x_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(x_test), batch_size=BATCH_SIZE)

for epoch in range(EPOCHS):
    autoencoder.train()
    train_loss = 0.0
    for (x,) in train_loader:
        x = x.to(device)
        loss = loss_fn(autoencoder(x), x)  # the target is the input itself

        opt.zero_grad()
        loss.backward()
        opt.step()
        train_loss += loss.item() * len(x)
    train_loss /= len(train_loader.dataset)

    autoencoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for (x,) in val_loader:
            x = x.to(device)
            val_loss += loss_fn(autoencoder(x), x).item() * len(x)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")
    writer.add_scalar("loss/train", train_loss, epoch)
    writer.add_scalar("loss/val", val_loss, epoch)

    if train_loss < best_loss:  # monitor="loss", save_best_only=True
        best_loss = train_loss
        torch.save(autoencoder.state_dict(), checkpoint_path)

writer.close()

In [ ]:
# Save the final models
# PyTorch saves the weights (state_dict); to reload, rebuild the model and
# call model.load_state_dict(torch.load(path))
torch.save(autoencoder.state_dict(), "./models/autoencoder.pt")
torch.save(encoder.state_dict(), "./models/encoder.pt")
torch.save(decoder.state_dict(), "./models/decoder.pt")

## 4. Reconstruct using the autoencoder <a name="reconstruct"></a>

In [ ]:
n_to_predict = 5000
example_images = x_test[:n_to_predict]
example_labels = y_test[:n_to_predict]

In [ ]:
def predict(model, inputs, batch_size=32):
    """Run a model over inputs in batches (like Keras' model.predict) and return a NumPy array."""
    model.eval()
    inputs = torch.as_tensor(inputs, dtype=torch.float32)
    with torch.no_grad():
        outputs = [model(batch.to(device)).cpu() for batch in inputs.split(batch_size)]
    return torch.cat(outputs).numpy()


predictions = predict(autoencoder, example_images)

print("Example real clothing items")
display(example_images)
print("Reconstructions")
display(predictions)

## 5. Embed using the encoder <a name="encode"></a>

In [ ]:
# Encode the example images
embeddings = predict(encoder, example_images)

In [ ]:
# Some examples of the embeddings
print(embeddings[:10])

In [ ]:
# Show the encoded points in 2D space
figsize = 8

plt.figure(figsize=(figsize, figsize))
plt.scatter(embeddings[:, 0], embeddings[:, 1], c="black", alpha=0.5, s=3)
plt.show()

In [ ]:
# Colour the embeddings by their label (clothing type - see table)
example_labels = y_test[:n_to_predict]

figsize = 8
plt.figure(figsize=(figsize, figsize))
plt.scatter(
    embeddings[:, 0],
    embeddings[:, 1],
    cmap="rainbow",
    c=example_labels,
    alpha=0.8,
    s=3,
)
plt.colorbar()
plt.show()

## 6. Generate using the decoder <a name="decode"></a>

In [ ]:
# Get the range of the existing embeddings
mins, maxs = np.min(embeddings, axis=0), np.max(embeddings, axis=0)

# Sample some points in the latent space
grid_width, grid_height = (6, 3)
sample = np.random.uniform(
    mins, maxs, size=(grid_width * grid_height, EMBEDDING_DIM)
)

In [ ]:
# Decode the sampled points
reconstructions = predict(decoder, sample)

In [ ]:
# Draw a plot of...
figsize = 8
plt.figure(figsize=(figsize, figsize))

# ... the original embeddings ...
plt.scatter(embeddings[:, 0], embeddings[:, 1], c="black", alpha=0.5, s=2)

# ... and the newly generated points in the latent space
plt.scatter(sample[:, 0], sample[:, 1], c="#00B0F0", alpha=1, s=40)
plt.show()

# Add underneath a grid of the decoded images
fig = plt.figure(figsize=(figsize, grid_height * 2))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i in range(grid_width * grid_height):
    ax = fig.add_subplot(grid_height, grid_width, i + 1)
    ax.axis("off")
    ax.text(
        0.5,
        -0.35,
        str(np.round(sample[i, :], 1)),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.imshow(reconstructions[i, 0], cmap="Greys")  # (C, H, W) -> first channel

In [ ]:
# Colour the embeddings by their label (clothing type - see table)
figsize = 12
grid_size = 15
plt.figure(figsize=(figsize, figsize))
plt.scatter(
    embeddings[:, 0],
    embeddings[:, 1],
    cmap="rainbow",
    c=example_labels,
    alpha=0.8,
    s=300,
)
plt.colorbar()

x = np.linspace(min(embeddings[:, 0]), max(embeddings[:, 0]), grid_size)
y = np.linspace(max(embeddings[:, 1]), min(embeddings[:, 1]), grid_size)
xv, yv = np.meshgrid(x, y)
xv = xv.flatten()
yv = yv.flatten()
grid = np.array(list(zip(xv, yv)))

reconstructions = predict(decoder, grid)
# plt.scatter(grid[:, 0], grid[:, 1], c="black", alpha=1, s=10)
plt.show()

fig = plt.figure(figsize=(figsize, figsize))
fig.subplots_adjust(hspace=0.4, wspace=0.4)
for i in range(grid_size**2):
    ax = fig.add_subplot(grid_size, grid_size, i + 1)
    ax.axis("off")
    ax.imshow(reconstructions[i, 0], cmap="Greys")